# Preprocessing Data from the AmbigNQ Dataset

1. Download dataset

In [ ]:
from datasets import load_dataset
dataset = load_dataset("sewon/ambig_qa", "light")

README.md:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

c:\Users\b-jtu\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\b-jtu\.cache\huggingface\hub\datasets--sewon--ambig_qa. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\b-jtu\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_dow

light/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.41MB            

light/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

light/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  367kB            

light/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10036 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2002 [00:00<?, ? examples/s]

In [2]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'annotations'],
        num_rows: 10036
    })
    validation: Dataset({
        features: ['id', 'question', 'annotations'],
        num_rows: 2002
    })
})


2. Preprocess data

In [12]:
import re

def preprocess_question(example):

    # Extract the original question.
    query = example["question"]

    # Extract the annotation fields.
    annotations = example["annotations"]

    # Initialize lists for extracted information.
    interpretations = []
    answers = []
    annotation_types = []

    # Extract the annotation types.
    types = annotations["type"]

    # Extract the original annotation answers.
    annotation_answers = annotations["answer"]

    # Extract the question-answer pairs.
    qa_pairs = annotations["qaPairs"]

    # Process each annotation type.
    for annotation_type in types:
        annotation_types.append(annotation_type)

    # Extract answers from the original annotations.
    for answer_group in annotation_answers:
        answers.extend(answer_group)

    # Process each question-answer pair.
    for pair in qa_pairs:

        # Extract the disambiguated questions.
        interpretations.extend(pair["question"])

        # Extract the corresponding answers.
        for answer_group in pair["answer"]:
            answers.extend(answer_group)

    # Remove duplicate interpretations and answers.
    interpretations = list(dict.fromkeys(interpretations))
    answers = list(dict.fromkeys(answers))

    # Normalize whitespace in the original question.
    query = re.sub(r"\s+", " ", query).strip()

    # Assign a preliminary ambiguity label.
    ambiguity_label = int("multipleQAs" in annotation_types)

    # Return the processed example.
    return {
        "id": example["id"],
        "query": query,
        "ambiguity_label": ambiguity_label,
        "annotation_type": annotation_types,
        "interpretations": interpretations,
        "answers": answers
    }

In [ ]:
#how processing works on one example
test_processed = preprocess_question(dataset["train"][0])
from pprint import pprint
pprint(test_processed)

{'ambiguity_label': 1,
 'annotation_type': ['multipleQAs'],
 'answers': ['April 19, 1987', 'December 17, 1989'],
 'id': '-4469503464110108318',
 'interpretations': ['When did the Simpsons first air on television as an '
                     'animated short on the Tracey Ullman Show?',
                     'When did the Simpsons first air as a half-hour prime '
                     'time show?'],
 'query': 'When did the simpsons first air on television?'}


In [ ]:
#process train and test
train_processed = [
    preprocess_question(example)
    for example in dataset["train"]
]
validation_processed = [
    preprocess_question(example)
    for example in dataset["validation"]
]

3. Save data

In [ ]:
import pandas as pd
train_df = pd.DataFrame(train_processed)
validation_df = pd.DataFrame(validation_processed)

In [16]:
# Count the number of clear and ambiguous questions.
print(train_df["ambiguity_label"].value_counts())

# Display the label proportions.
print(train_df["ambiguity_label"].value_counts(normalize=True))

ambiguity_label
0    5287
1    4749
Name: count, dtype: int64
ambiguity_label
0    0.526804
1    0.473196
Name: proportion, dtype: float64


In [18]:
# Display the annotation type distribution.
print(train_df["annotation_type"].explode().value_counts())

# Compare annotation types across ambiguity labels.
print(
    train_df.explode("annotation_type")
    .groupby(["ambiguity_label", "annotation_type"])
    .size()
    .unstack(fill_value=0)
)

annotation_type
singleAnswer    5459
multipleQAs     4792
Name: count, dtype: int64
annotation_type  multipleQAs  singleAnswer
ambiguity_label                           
0                          0          5449
1                       4792            10


In [ ]:
import os
os.makedirs("data/processed", exist_ok=True)
train_df.to_csv("data/processed/train.csv", index=False)
validation_df.to_csv("data/processed/validation.csv", index=False)